In [ ]:
!pip install pandas torch transformers unsloth scikit-learn openpyxl


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 2.3 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.5/203.5 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.4/491.4 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 318.9/318.9 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.0/129.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.5/31.5 MB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 865.2/865.2 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.

In [ ]:
import torch
import pandas as pd
from transformers import AutoTokenizer
from unsloth import FastLanguageModel
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from difflib import SequenceMatcher


<ipython-input-2-7f32badd7f86>:4: UserWarning: WARNING: Unsloth should be imported before transformers to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:
# Define Hugging Face model path
hf_model_path = "deepanshumiglani0408/mistral-7b-instruct-hindi-qa"  # Your uploaded model

# Load Model from Hugging Face Hub
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=hf_model_path,
    max_seq_length=1024,  # Adjust based on your training
    dtype=torch.float16,  # Use fp16 for efficiency
    load_in_4bit=True,  # 4-bit quantization for speed
)

# Prepare Model for Fast Inference
FastLanguageModel.for_inference(model)

print("✅ Model loaded successfully from Hugging Face!")


==((====))==  Unsloth 2025.4.3: Fast Mistral patching. Transformers: 4.51.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.3.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.30. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/4.13G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/155 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.13k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/438 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/168M [00:00<?, ?B/s]

Unsloth 2025.4.3 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


✅ Model loaded successfully from Hugging Face!


In [ ]:
# Load Excel file
df = pd.read_excel("HindGK Test Dataset.xlsx")  # Ensure file is uploaded

# Keep only first 500 rows
df = df.head(500)

# Ensure correct column names
df = df.rename(columns={"question": "Question", "answer": "Actual_Answer"})

# Display sample
df.head()


FileNotFoundError: [Errno 2] No such file or directory: 'HindGK Test Dataset.xlsx'

In [ ]:
!pip install tqdm



In [ ]:
def generate_answer(question):
    system_prompt = "आप एक सहायक AI हैं जो हिंदी में सवालों के जवाब देती है। आपको स्पष्ट, सटीक और सहायक जवाब देने चाहिए।"
    # Tokenize input
    inputs = tokenizer(
        [
            f"<|start_header_id|>system<|end_header_id|>{system_prompt}<|eot_id|>"
            f"<|start_header_id|>user<|end_header_id|>{question}<|end_header_id|>"
        ],
        return_tensors="pt"
    ).to("cuda" if torch.cuda.is_available() else "cpu")
    # Generate response
    outputs = model.generate(**inputs, max_new_tokens=256, use_cache=True)
    # Decode response
    full_response = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]

    # Extract only the main answer
    # Find position after the question and before any EOT markers
    try:
        # This assumes the model response comes after the header tags for user
        response_part = full_response.split("<|end_header_id|>")[-1].split("<|eot_id|>")[0]
        # Clean any trailing newlines or extra spaces
        clean_answer = response_part.strip()
        return clean_answer
    except:
        # Fallback in case the splitting doesn't work as expected
        return full_response.strip()

In [ ]:
# Initialize progress bar
from tqdm import tqdm
tqdm.pandas(desc="⏳ Generating AI Predictions...")

# Apply function to get AI-generated answers
df["Predicted_Answer"] = df["Question"].progress_apply(generate_answer)

# Display predictions
df.head(10)


⏳ Generating AI Predictions...: 100%|██████████| 500/500 [1:28:48<00:00, 10.66s/it]


,Question,Actual_Answer,Predicted_Answer
0,पहले सफल ऑटोमोबाइल का आविष्कार किसने किया?,कार्ल बेंज को पहले सफल ऑटोमोबाइल का आविष्कार क...,किंग लेविस ने पहले सफल ऑटोमोबाइल का आविष्कार क...
1,डीएनए की संरचना की खोज किसने की?,जेम्स वॉटसन और फ्रांसिस क्रिक को डीएनए की संरच...,जेनेटिक पारमा डीएनए की संरचना की खोज की।
2,टेलीफोन का आविष्कार किसने किया?,अलेक्जेंडर ग्राहम बेल को टेलीफोन का आविष्कार क...,अलेक्संडर ग्रेहम बेल टेलीफोन के आविष्कार के लि...
3,पेनिसिलिन की खोज किसने की?,अलेक्जेंडर फ्लेमिंग को पेनिसिलिन की खोज करने क...,एक ब्रिटिश चिकित्सक ने पेनिसिलिन की खोज की।<|e...
4,पहले सफल हवाई जहाज का आविष्कार किसने किया?,"राइट ब्रदर्स, ऑरविले और विल्बर को पहले सफल हवा...",ऑटो किरल पहले सफल हवाई जहाज क
5,परमाणु संरचना के सिद्धांतों की खोज किसने की?,"जे.जे. थॉमसन, अर्नेस्ट रदरफोर्ड, और नील्स बोह्...",एलेन हार्डिंग
6,पहले सफल कंप्यूटर का आविष्कार किसने किया?,जॉन अतानासॉफ और क्लिफोर्ड बेरी को पहले सफल कंप...,चेल नाटे पहले सफल कंप्यूटर क
7,प्राकृतिक चयन द्वारा विकास के सिद्धांतों की खो...,चार्ल्स डार्विन को प्राकृतिक चयन द्वारा विकास ...,चार्ल्स डार्विन ने प्राकृतिक चयन द्वारा विकास ...
8,गुरुत्वाकर्षण के सिद्धांतों की खोज किसने की?,सर आइजैक न्यूटन को गुरुत्वाकर्षण के सिद्धांतों...,इस्टैन न्यूटन गुरुत्वाकर्षण के सिद्धांतों की ख...
9,पहले सफल स्टीम इंजन का आविष्कार किसने किया?,जेम्स वाट को पहले सफल स्टीम इंजन का आविष्कार क...,इस्टैन न्यूटन पहले सफल स्टीम इंजन का आविष्कार ...


In [ ]:
# ✅ Save predictions to a CSV file
df.to_csv("predictions(pre-trained+finetuned).csv", index=False, encoding="utf-8-sig")

print("✅ Predictions saved successfully as 'predictions.csv' 🎯")


✅ Predictions saved successfully as 'predictions.csv' 🎯


In [ ]:
# Import necessary libraries
import numpy as np
from difflib import SequenceMatcher
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from tqdm import tqdm

def similarity(a, b):
    """Compute similarity score between actual and predicted answers."""
    return SequenceMatcher(None, str(a), str(b)).ratio()  # Returns score between 0-1

# Add progress bar
tqdm.pandas(desc="🔍 Calculating Similarity...")
df["Similarity_Score"] = df.progress_apply(lambda x: similarity(x["Actual_Answer"], x["Predicted_Answer"]), axis=1)

# Display similarity scores to see what's happening
print("Sample of similarity scores:")
print(df[["Actual_Answer", "Predicted_Answer", "Similarity_Score"]].head(10))

# ✅ Try a lower threshold - 0.7 might be too high for your data
df["Correct"] = df["Similarity_Score"] > 0.3

# Display results
print("\nAfter applying threshold:")
print(df[["Similarity_Score", "Correct"]].head(10))

# Simple accuracy calculation (percentage of correct predictions)
accuracy = df["Correct"].mean()
print(f"\n📊 Accuracy: {accuracy:.4f}")

# Count correct and incorrect predictions
correct_count = df["Correct"].sum()
total_count = len(df)
incorrect_count = total_count - correct_count

print(f"\n✅ Correct Predictions: {correct_count} ({correct_count/total_count:.2%})")
print(f"❌ Incorrect Predictions: {incorrect_count} ({incorrect_count/total_count:.2%})")

# If you still want precision, recall, and F1 score calculation
# Create binary arrays for actual predictions (your model results)
y_pred = df["Correct"].astype(int).values

# Create binary array for expected results (all should be correct)
y_true = np.ones(len(df), dtype=int)

# Calculate metrics
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

print(f"🎯 Precision: {precision:.4f}")
print(f"📢 Recall: {recall:.4f}")
print(f"🏆 F1-Score: {f1:.4f}")

🔍 Calculating Similarity...: 100%|██████████| 500/500 [00:00<00:00, 2054.55it/s]

Sample of similarity scores:
                                       Actual_Answer  \
0  कार्ल बेंज को पहले सफल ऑटोमोबाइल का आविष्कार क...   
1  जेम्स वॉटसन और फ्रांसिस क्रिक को डीएनए की संरच...   
2  अलेक्जेंडर ग्राहम बेल को टेलीफोन का आविष्कार क...   
3  अलेक्जेंडर फ्लेमिंग को पेनिसिलिन की खोज करने क...   
4  राइट ब्रदर्स, ऑरविले और विल्बर को पहले सफल हवा...   
5  जे.जे. थॉमसन, अर्नेस्ट रदरफोर्ड, और नील्स बोह्...   
6  जॉन अतानासॉफ और क्लिफोर्ड बेरी को पहले सफल कंप...   
7  चार्ल्स डार्विन को प्राकृतिक चयन द्वारा विकास ...   
8  सर आइजैक न्यूटन को गुरुत्वाकर्षण के सिद्धांतों...   
9  जेम्स वाट को पहले सफल स्टीम इंजन का आविष्कार क...   

                                    Predicted_Answer  Similarity_Score  
0  किंग लेविस ने पहले सफल ऑटोमोबाइल का आविष्कार क...          0.434783  
1           जेनेटिक पारमा डीएनए की संरचना की खोज की।          0.358696  
2  अलेक्संडर ग्रेहम बेल टेलीफोन के आविष्कार के लि...          0.488263  
3  एक ब्रिटिश चिकित्सक ने पेनिसिलिन की खोज की।<|e...          

In [ ]:
# ✅ Ask a Question & Get AI Answer
system_prompt = "आप एक सहायक AI हैं जो हिंदी में सवालों के जवाब देती है। आपको स्पष्ट, सटीक और सहायक जवाब देने चाहिए।"

while True:
    question = input("अपना प्रश्न हिंदी में टाइप करें (या 'exit' टाइप करें): ")
    if question.lower() == "exit":
        print("🚀 सत्र समाप्त! धन्यवाद!")
        break

    inputs = tokenizer(
        [
            f"<|start_header_id|>system<|end_header_id|>{system_prompt}<|eot_id|>"
            f"<|start_header_id|>user<|end_header_id|>{question}<|eot_id|>"
            f"<|start_header_id|>assistant<|end_header_id|>"  # This ensures AI only responds
        ],
        return_tensors="pt"
    ).to("cuda" if torch.cuda.is_available() else "cpu")

    outputs = model.generate(**inputs, max_new_tokens=256, use_cache=True)

    # Extract only the AI's response
    full_response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Clean the response to get only the main answer
    try:
        # First try to get everything after the assistant header
        response = full_response.split("<|start_header_id|>assistant<|end_header_id|>")[-1]

        # Remove any trailing EOT tags or other special tokens
        if "<|eot_id|>" in response:
            response = response.split("<|eot_id|>")[0]

        # Clean up any extra whitespace
        clean_response = response.strip()
        print("\n🤖 AI का उत्तर:", clean_response)
    except:
        # Fallback if the splitting doesn't work
        print("\n🤖 AI का उत्तर:", full_response.strip())

अपना प्रश्न हिंदी में टाइप करें (या 'exit' टाइप करें): भारत की राजधानी क्या है?

🤖 AI का उत्तर: भारत की राजधानी नई दिल्ली है। \ n
अपना प्रश्न हिंदी में टाइप करें (या 'exit' टाइप करें): भारत की स्वतंत्रता किस वर्ष मिली थी?

🤖 AI का उत्तर: भारत की स्वतंत्रता 15 अगस्त 1947 में मिली। \ n


In [ ]:
#भारत की राजधानी क्या है?
#भारत का राष्ट्रीय पक्षी कौन सा है?
#भारत की स्वतंत्रता किस वर्ष मिली थी?
